In [70]:
import pandas as pd
import sqlite3
from datetime import datetime

from zoneinfo import ZoneInfo

# Test case: Payments reconciliation — June 2026

## Providers

Lumo Wellness sells app subscriptions through five payment providers: PayPal (separate US and EU accounts), dLocal (Latin America), Adyen, and Google Play.

In [187]:
# PayPal US account activity export
df_paypal_us_activity_20260601_20260703 = pd.read_csv("raw/paypal_us_activity_20260601_20260703.csv") 
df_paypal_us_activity_20260601_20260703.columns = df_paypal_us_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_paypal_us_activity_20260601_20260703["datetime_utc"] = df_paypal_us_activity_20260601_20260703.apply(
    lambda row: pd.to_datetime(f"{row['date']} {row['time']}", format="%m/%d/%Y %H:%M:%S").tz_localize(ZoneInfo(row["time_zone"])).tz_convert("UTC")
    , axis=1)
df_paypal_us_activity_20260601_20260703.head()


,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,06/01/2026,07:12:11,GMT,Taylor Brooks,7FTLSR4XAT5TKVLKF,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500884,2026-06-01 07:12:11+00:00
1,06/01/2026,07:54:32,GMT,Morgan Hayes,7R82A7SLSXJAD72J1,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500132,2026-06-01 07:54:32+00:00
2,06/01/2026,07:59:18,GMT,Drew Bennett,S1VNZT7ACC5E6HNYD,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500591,2026-06-01 07:59:18+00:00
3,06/01/2026,08:03:29,GMT,Riley Carter,BTDR7VQTQ8G4GTMGS,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500479,2026-06-01 08:03:29+00:00
4,06/01/2026,08:16:21,GMT,Casey Nguyen,NFQZ142PMP4P1BC2Q,NaN,Website Payment,Completed,USD,24.99,-1.36,23.63,ORD-500005,2026-06-01 08:16:21+00:00


In [186]:
# PayPal EU account activity export
# semicolon, cp1252 (Windows-1252)
df_paypal_eu_activity_20260601_20260703 = pd.read_csv("raw/paypal_eu_activity_20260601_20260703.csv", sep=";", encoding="cp1252") 
df_paypal_eu_activity_20260601_20260703.columns = df_paypal_eu_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_paypal_eu_activity_20260601_20260703["datetime_utc"] = df_paypal_eu_activity_20260601_20260703.apply(
    lambda row: pd.to_datetime(f"{row['date']} {row['time']}", format="%d/%m/%Y %H:%M:%S").tz_localize(ZoneInfo(row["time_zone"])).tz_convert("UTC")
    , axis=1)
df_paypal_eu_activity_20260601_20260703.head()

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,06/03/2026,12:20:00,GMT,Jürgen Müller,N2D9QMQNH8VUEES5X,NaN,Website Payment,Completed,EUR,"9,99","-0,64","9,35",ORD-507782,2026-03-06 12:20:00+00:00
1,01/06/2026,08:17:07,GMT,Céline Marchand,ZEW9RTFHKTBFQT5E3,NaN,Website Payment,Completed,GBP,"8,99","-0,56","8,43",ORD-502669,2026-06-01 08:17:07+00:00
2,01/06/2026,08:35:02,GMT,José García,0DRQ0928NVSA52MA3,NaN,Website Payment,Denied,GBP,"4,49","0,00","0,00",ORD-503463,2026-06-01 08:35:02+00:00
3,01/06/2026,08:48:13,GMT,Nuño Ibáñez,P54742W2ULA8YDR0L,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502556,2026-06-01 08:48:13+00:00
4,01/06/2026,09:07:25,GMT,Søren Kjær,JR1SUR8G07W17ZPNR,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502537,2026-06-01 09:07:25+00:00


In [188]:
# Adyen payment accounting report
df_adyen_payment_accounting_20260601_20260703 = pd.read_csv("raw/adyen_payment_accounting_20260601_20260703.csv") 
df_adyen_payment_accounting_20260601_20260703.columns = df_adyen_payment_accounting_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_adyen_payment_accounting_20260601_20260703["datetime_utc"] = df_adyen_payment_accounting_20260601_20260703.apply(
    lambda row: pd.Timestamp(f"{row['creation_date']}").tz_localize(ZoneInfo(row["timezone"])).tz_convert("UTC")
    , axis=1)
df_adyen_payment_accounting_20260601_20260703.head()

,company_account,merchant_account,psp_reference,merchant_reference,record_type,creation_date,timezone,gross_currency,gross_debit,gross_credit,commission,markup,net_currency,net_debit,net_credit,batch_number,datetime_utc
0,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Authorised,2026-06-01 09:13:28,Europe/Amsterdam,USD,NaN,24.99,NaN,NaN,USD,NaN,NaN,23,2026-06-01 07:13:28+00:00
1,LumoGroup,LumoECOM_EU,8620000000000500,ORD-504450,Authorised,2026-06-01 09:15:43,Europe/Amsterdam,EUR,NaN,9.99,NaN,NaN,EUR,NaN,NaN,23,2026-06-01 07:15:43+00:00
2,LumoGroup,LumoECOM_EU,8620000000000115,ORD-504065,Authorised,2026-06-01 09:18:57,Europe/Amsterdam,EUR,NaN,24.99,NaN,NaN,EUR,NaN,NaN,23,2026-06-01 07:18:57+00:00
3,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Settled,2026-06-01 09:36:28,Europe/Amsterdam,USD,NaN,24.99,0.47,0.15,USD,NaN,24.37,23,2026-06-01 07:36:28+00:00
4,LumoGroup,LumoAPP_US,8620000000000575,ORD-504525,Authorised,2026-06-01 09:41:36,Europe/Amsterdam,USD,NaN,9.99,NaN,NaN,USD,NaN,NaN,23,2026-06-01 07:41:36+00:00


In [189]:
# dLocal transactions export
df_dlocal_transactions_20260601_20260703 = pd.read_csv("raw/dlocal_transactions_20260601_20260703.csv") 
df_dlocal_transactions_20260601_20260703.head()

,transaction_id,invoice_id,transaction_type,status,created_at_utc,country,currency,currency_exponent,local_amount,fx_rate,usd_amount,fee_usd
0,DL-90001309,ORD-507781,PAYMENT,PAID,2026-06-01 00:05:00,MX,MXN,2,12900,18.1548,7.11,0.32
1,DL-90001242,ORD-506937,PAYMENT,REJECTED,2026-06-01 07:25:52,CO,COP,2,19990000,3984.0637,50.17,2.26
2,DL-90000404,ORD-506099,PAYMENT,PAID,2026-06-01 08:17:24,MX,MXN,2,12900,18.1772,7.10,0.32
3,DL-90000750,ORD-506445,PAYMENT,PAID,2026-06-01 08:27:11,CO,COP,2,2990000,3984.0637,7.50,0.34
4,DL-90001024,ORD-506719,PAYMENT,PAID,2026-06-01 08:29:13,BR,BRL,2,3490,5.4195,6.44,0.29


In [190]:
# Google Play earnings report, June
# line 1 is a report header, data starts on line 2
df_google_play_earnings_202606 = pd.read_csv("raw/google_play_earnings_202606.csv", skiprows=1, header=0)
df_google_play_earnings_202606.columns = df_google_play_earnings_202606.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)

df_google_play_earnings_202606["datetime_utc"] = df_google_play_earnings_202606.apply(
    lambda row: pd.Timestamp(f"{row['transaction_date']} {row['transaction_time']}").tz_localize(ZoneInfo("America/Los_Angeles")).tz_convert("UTC")
    , axis=1)
df_google_play_earnings_202606.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency,datetime_utc
0,"Jun 1, 2026",00:28:46,Charge,sub_annual,US,USD,59.99,1.0,USD,59.99,2026-06-01 07:28:46+00:00
1,"Jun 1, 2026",00:28:46,Google fee,sub_annual,US,USD,-9.00,1.0,USD,-9.00,2026-06-01 07:28:46+00:00
2,"Jun 1, 2026",00:53:08,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99,2026-06-01 07:53:08+00:00
3,"Jun 1, 2026",00:53:08,Google fee,sub_monthly,US,USD,-1.50,1.0,USD,-1.50,2026-06-01 07:53:08+00:00
4,"Jun 1, 2026",01:35:01,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99,2026-06-01 08:35:01+00:00


In [191]:
# Google Play earnings report, July 1–3
df_google_play_earnings_202607_partial = pd.read_csv("raw/google_play_earnings_202607_partial.csv", skiprows=1, header=0) 
df_google_play_earnings_202607_partial.columns = df_google_play_earnings_202607_partial.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)

df_google_play_earnings_202607_partial["datetime_utc"] = df_google_play_earnings_202607_partial.apply(
    lambda row: pd.Timestamp(f"{row['transaction_date']} {row['transaction_time']}").tz_localize(ZoneInfo("America/Los_Angeles")).tz_convert("UTC")
    , axis=1)
df_google_play_earnings_202607_partial.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency,datetime_utc
0,"Jul 1, 2026",02:40:10,Charge refund,sub_monthly,DE,EUR,-9.99,1.085329,USD,-10.84,2026-07-01 09:40:10+00:00
1,"Jul 1, 2026",03:50:07,Charge,sub_monthly,DE,EUR,9.99,1.085329,USD,10.84,2026-07-01 10:50:07+00:00
2,"Jul 1, 2026",03:50:07,Google fee,sub_monthly,DE,EUR,-1.50,1.085329,USD,-1.63,2026-07-01 10:50:07+00:00
3,"Jul 1, 2026",05:43:32,Charge,sub_monthly,US,USD,9.99,1.000000,USD,9.99,2026-07-01 12:43:32+00:00
4,"Jul 1, 2026",05:43:32,Google fee,sub_monthly,US,USD,-1.50,1.000000,USD,-1.50,2026-07-01 12:43:32+00:00


## Backend

payment_engine_log.csv is an extract from the payment backend — the engine that initiates and tracks every transaction attempt, applies FX conversion, and is treated internally as the source of truth. The other files are data exports from each payment provider’s admin panel — it’s what the PSPs themselves say happened.

In [192]:
# Payment backend (engine) log extract, all providers
df_payment_engine_log = pd.read_csv("raw/payment_engine_log.csv") 
df_payment_engine_log.head()

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107955,ORD-507781,SALE,settled,dlocal,DL-90001309,sub_monthly,MX,MXN,129.00,0.055082,2026-05-29,7.11,2026-05-31T23:59:50Z,2026-06-01T00:05:00Z
1,TXN-100884,ORD-500884,SALE,settled,paypal_us,7FTLSR4XAT5TKVLKF,sub_monthly,US,USD,9.99,1.000000,2026-06-01,9.99,2026-06-01T07:03:11Z,2026-06-01T07:12:11Z
2,TXN-106445,ORD-506445,SALE,settled,dlocal,DL-90000750,sub_monthly,CO,COP,29900.00,0.000251,2026-06-01,7.50,2026-06-01T07:11:11Z,2026-06-01T08:27:11Z
3,TXN-106099,ORD-506099,SALE,settled,dlocal,DL-90000404,sub_monthly,MX,MXN,129.00,0.055014,2026-06-01,7.10,2026-06-01T07:11:24Z,2026-06-01T08:17:24Z
4,TXN-105905,ORD-505905,SALE,settled,dlocal,DL-90000210,sub_quarterly,BR,BRL,89.90,0.184519,2026-06-01,16.59,2026-06-01T07:12:27Z,2026-06-01T08:53:27Z


In [193]:
# Daily FX rates to USD (usd_rate = USD per 1 unit of currency)
df_fx_rates = pd.read_csv("raw/fx_rates.csv") 
df_fx_rates.head()

,rate_date,currency,usd_rate,published_at
0,2026-05-25,ARS,0.000839,2026-05-25T16:00:00Z
1,2026-05-25,BRL,0.184871,2026-05-25T16:00:00Z
2,2026-05-25,CLP,0.001060,2026-05-25T16:00:00Z
3,2026-05-25,COP,0.000248,2026-05-25T16:00:00Z
4,2026-05-25,EUR,1.093525,2026-05-25T16:00:00Z


In [194]:
# Contracted provider fees, where we have a contract on file
df_fee_schedule = pd.read_csv("raw/fee_schedule.csv") 
df_fee_schedule.head()

,psp,account,valid_from,percent_fee,fixed_fee,fixed_fee_currency
0,adyen,LumoECOM_EU,2026-01-01,2.5,0.0,NaN
1,adyen,LumoAPP_US,2026-01-01,2.5,0.0,NaN
2,google_play,LumoPlayMain,2026-01-01,15.0,0.0,NaN


In [195]:
# connect to a database
conn = sqlite3.connect("db_simple_life.db")

df_payment_engine_log.to_sql('payment_engine_log', conn, if_exists="replace", index=False)

df_paypal_us_activity_20260601_20260703.to_sql('paypal_us_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_paypal_eu_activity_20260601_20260703.to_sql('paypal_eu_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_adyen_payment_accounting_20260601_20260703.to_sql('adyen_payment_accounting_20260601_20260703', conn, if_exists="replace", index=False)
df_dlocal_transactions_20260601_20260703.to_sql('dlocal_transactions_20260601_20260703', conn, if_exists="replace", index=False)
df_google_play_earnings_202606.to_sql('google_play_earnings_202606', conn, if_exists="replace", index=False)
df_google_play_earnings_202607_partial.to_sql('google_play_earnings_202607_partial', conn, if_exists="replace", index=False)
df_fx_rates.to_sql('fx_rates', conn, if_exists="replace", index=False)
df_fee_schedule.to_sql('fee_schedule', conn, if_exists="replace", index=False)


3

Every month, Finance Analyst must prove that the payment backend and the provider exports tell the same story, and explain every difference in dollars if it’s not the case. That is your task, for the reporting period June 2026.

The engine-log extract and the provider exports intentionally cover a few days around the month boundary as well.


## Your task
- Reconcile each provider export against the payment engine log for June 2026. Quantify all discrepancies — in row counts and in USD.
- Classify every discrepancy by root cause, using your own taxonomy.
- Deliver a summary at provider × cause grain: row count, absolute USD impact, signed USD impact, and each cause’s share of the total discrepancy in %.
- Where fee_schedule.csv is applicable to  a provider, reconcile the reported fees against the contracted rates.
- Report the total discrepancy and show what remains unexplained after your classification (the residual). A correct analysis explains all the discrepancies.

## Evaluation notes
- Use DB of your choice (but tell us which), we assess the logic, not the dialect.
- When the data is ambiguous, make a call and document it. Documented assumptions count in your favor.
- Every dollar of difference between the payment backend and the providers is either matched, explained with a cause, or explicitly listed as unexplained — and the three add up.
- The reconciliation summary should read like something you could hand to a CFO: numbers first, caveats and issues stated.

## What to hand back
- Your SQL
- The summary from step 3, as a query result (CSV).
- A short written note: your assumptions, data-quality findings, and action items you would take to fix upstream.


In [78]:
# query0 = """
# SELECT *
# -- FROM payment_engine_log -- +
# -- FROM paypal_us_activity_20260601_20260703 -- +
# -- FROM paypal_eu_activity_20260601_20260703 -- +
# -- FROM adyen_payment_accounting_20260601_20260703 -- +
# -- FROM dlocal_transactions_20260601_20260703 -- +
# FROM google_play_earnings_202606 -- +
# -- FROM google_play_earnings_202607_partial -- +
# -- FROM fx_rates -- +
# -- FROM fee_schedule -- +
# LIMIT 100;
# """
# result0 = pd.read_sql(query0, conn)
# result0

In [340]:
query_backend_pivot = """
SELECT 
    psp, 
    country, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
GROUP BY 1,2,3
LIMIT 100;
"""
result_backend_pivot = pd.read_sql(query_backend_pivot, conn)
result_backend_pivot

,psp,country,currency,amount_local,amount_usd
0,adyen,DE,EUR,10269.94,11174.15
1,adyen,FR,EUR,4924.89,5358.69
2,adyen,GB,GBP,4941.21,6383.63
3,adyen,NL,EUR,6646.49,7236.50
4,adyen,US,USD,6941.67,6941.67
5,dlocal,AR,ARS,1300490.50,1086.98
6,dlocal,BR,BRL,40246.37,7491.96
7,dlocal,CL,CLP,2057005.00,2184.89
8,dlocal,CO,COP,7891100.00,201797.80
9,dlocal,MX,MXN,77300.00,5019.47


In [341]:
query_backend_pivot_usd = """
SELECT 
    psp
    , sum(amount_usd) as amount_usd
    , count(*) as backend_row_cnt
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
GROUP BY 1
"""
result_backend_pivot_usd = pd.read_sql(query_backend_pivot_usd, conn)
result_backend_pivot_usd

,psp,amount_usd,backend_row_cnt
0,adyen,37094.64,1736
1,dlocal,217581.10,1278
2,google_play,15116.05,776
3,paypal_eu,40411.14,1831
4,paypal_us,40314.22,2093


- payment_engine_log - created_at_utc
- paypal (eu, us) - GMT
- adyen - Europe/Amsterdam
- dlocal - created_at_utc
- google play - All dates and times are in America/Los_Angeles

### FX rates modification

Check if each currency has the only row for each day:

In [245]:
fx_rates = """
SELECT rate_date, count(*) as cnt
FROM fx_rates
group by 1
order by 1 desc 
"""
df_fx_rates = pd.read_sql(fx_rates, conn)
df_fx_rates

,rate_date,cnt
0,2026-07-03,7
1,2026-07-02,7
2,2026-07-01,7
3,2026-06-30,7
4,2026-06-29,7
5,2026-06-26,7
6,2026-06-25,7
7,2026-06-24,7
8,2026-06-23,7
9,2026-06-22,7


There is a later version for BRL at Jun 10:

In [220]:
fx_rates = """
SELECT *
FROM fx_rates
where rate_date = '2026-06-10'
"""
df_fx_rates = pd.read_sql(fx_rates, conn)
df_fx_rates

,rate_date,currency,usd_rate,published_at
0,2026-06-10,ARS,0.000838,2026-06-10T16:00:00Z
1,2026-06-10,BRL,0.185352,2026-06-10T16:00:00Z
2,2026-06-10,BRL,0.190542,2026-06-12T09:00:00Z
3,2026-06-10,CLP,0.001068,2026-06-10T16:00:00Z
4,2026-06-10,COP,0.000248,2026-06-10T16:00:00Z
5,2026-06-10,EUR,1.090715,2026-06-10T16:00:00Z
6,2026-06-10,GBP,1.288904,2026-06-10T16:00:00Z
7,2026-06-10,MXN,0.055489,2026-06-10T16:00:00Z


Exclude it:

In [221]:
fx_rates = """
SELECT rate_date, count(*) as cnt
FROM (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
)
where rn=1
group by 1
order by 1 desc 
"""
df_fx_rates = pd.read_sql(fx_rates, conn)
df_fx_rates

,rate_date,cnt
0,2026-07-03,7
1,2026-07-02,7
2,2026-07-01,7
3,2026-06-30,7
4,2026-06-29,7
5,2026-06-26,7
6,2026-06-25,7
7,2026-06-24,7
8,2026-06-23,7
9,2026-06-22,7


There is no FX data in weekends: 
- Jun 6-7
- Jun 13-14
- Jun 19 -- ???
- Jun 20-21
- Jun 27-28

How to add next_rate_date to cover these days in joins:

In [243]:
fx_rates1 = """
WITH fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
)

select *
from fx_rates_2
"""
df_fx_rates1 = pd.read_sql(fx_rates1, conn)
df_fx_rates1

,rate_date,currency,usd_rate,published_at,rn,next_rate_date
0,2026-05-25,ARS,0.000839,2026-05-25T16:00:00Z,1,2026-05-26
1,2026-05-26,ARS,0.000840,2026-05-26T16:00:00Z,1,2026-05-27
2,2026-05-27,ARS,0.000839,2026-05-27T16:00:00Z,1,2026-05-28
3,2026-05-28,ARS,0.000842,2026-05-28T16:00:00Z,1,2026-05-29
4,2026-05-29,ARS,0.000844,2026-05-29T16:00:00Z,1,2026-06-01
...,...,...,...,...,...,...
198,2026-06-29,MXN,0.055211,2026-06-29T16:00:00Z,1,2026-06-30
199,2026-06-30,MXN,0.055059,2026-06-30T16:00:00Z,1,2026-07-01
200,2026-07-01,MXN,0.055003,2026-07-01T16:00:00Z,1,2026-07-02
201,2026-07-02,MXN,0.054896,2026-07-02T16:00:00Z,1,2026-07-03


old version:

In [231]:
fx_rates2 = """
WITH RECURSIVE dates(rate_date) AS (
    SELECT MIN(rate_date)
    FROM (
        select *
            , row_number() over (partition by rate_date, currency order by published_at desc) as rn
        from fx_rates
    )
    where rn=1
    UNION ALL
    SELECT date(rate_date, '+1 day')
    FROM dates
    WHERE rate_date < (SELECT MAX(rate_date) FROM fx_rates)
),

currencies AS (
    SELECT DISTINCT currency
    FROM fx_rates
),

fx_rates_upd AS (
    SELECT
        dates.rate_date,
        c.currency,
        (
            SELECT t.usd_rate
            FROM (
                select *
                    , row_number() over (partition by rate_date, currency order by published_at desc) as rn
                from fx_rates
            ) t
            WHERE t.rate_date <= dates.rate_date
                AND t.currency = c.currency
                AND t.usd_rate IS NOT NULL
                AND t.rn=1
            ORDER BY currency, t.rate_date DESC
            LIMIT 1
        ) AS usd_rate
    FROM dates
    CROSS JOIN currencies c
    where dates.rate_date >= '2026-06-01'
    ORDER BY c.currency, dates.rate_date
)
select *
from fx_rates_upd
;
"""
df_fx_rates2 = pd.read_sql(fx_rates2, conn)
df_fx_rates2.head(15)

,rate_date,currency,usd_rate
0,2026-06-01,ARS,0.000843
1,2026-06-02,ARS,0.000844
2,2026-06-03,ARS,0.000841
3,2026-06-04,ARS,0.000841
4,2026-06-05,ARS,0.000840
5,2026-06-06,ARS,0.000840
6,2026-06-07,ARS,0.000840
7,2026-06-08,ARS,0.000841
8,2026-06-09,ARS,0.000839
9,2026-06-10,ARS,0.000838


### Duplicates check

In [282]:
query_duplicates = """
WITH orig as (
    select *
    from dlocal_transactions_20260601_20260703
),

sub as (
    select distinct *
    from orig
)

SELECT count(*) as cnt 
FROM orig
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
--WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'

UNION ALL

SELECT count(*) as cnt 
FROM sub
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
--WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'

"""
result_duplicates = pd.read_sql(query_duplicates, conn)
result_duplicates

,cnt
0,1279
1,1278


- payment_engine_log - no
- paypal_eu_activity_20260601_20260703 - no
- paypal_us_activity_20260601_20260703 - yes (1 row)
- adyen_payment_accounting_20260601_20260703 - no
- dlocal_transactions_20260601_20260703 - yes (1 row)
- google_play_earnings_202606 - no
- google_play_earnings_202607_partial - no

## Providers vs Backend Reconciliation

### Paypal EU

In [239]:
query_paypal = """
WITH RECURSIVE dates(rate_date) AS (
    SELECT MIN(rate_date)
    FROM (
        select *
            , row_number() over (partition by rate_date, currency order by published_at desc) as rn
        from fx_rates
    )
    where rn=1
    UNION ALL
    SELECT date(rate_date, '+1 day')
    FROM dates
    WHERE rate_date < (SELECT MAX(rate_date) FROM fx_rates)
),

currencies AS (
    SELECT DISTINCT currency
    FROM fx_rates
),

fx_rates_upd AS (
    SELECT
        dates.rate_date,
        c.currency,
        (
            SELECT t.usd_rate
            FROM (
                select *
                    , row_number() over (partition by rate_date, currency order by published_at desc) as rn
                from fx_rates
            ) t
            WHERE t.rate_date <= dates.rate_date
                AND t.currency = c.currency
                AND t.usd_rate IS NOT NULL
                AND t.rn=1
            ORDER BY currency, t.rate_date DESC
            LIMIT 1
        ) AS usd_rate
    FROM dates
    CROSS JOIN currencies c
    where dates.rate_date >= '2026-06-01'
    ORDER BY c.currency, dates.rate_date
),

paypal as (
    SELECT 
        pp.type,
        pp.status,
        pp.currency				
        , sum(CAST(REPLACE(pp.gross, ',', '.') AS REAL)) as gross
        , sum(CAST(REPLACE(pp.fee, ',', '.') AS REAL)) as fee
        , sum(CAST(REPLACE(pp.net, ',', '.') AS REAL)) as net
        , sum(ROUND(CAST(REPLACE(pp.gross, ',', '.') AS REAL) * fx.usd_rate, 2)) as gross_usd
        , sum(ROUND(CAST(REPLACE(pp.net, ',', '.') AS REAL) * fx.usd_rate, 2)) as net_usd
        , count(*) as provider_row_cnt
    FROM paypal_eu_activity_20260601_20260703 pp
    LEFT JOIN fx_rates_upd fx on fx.rate_date = date(pp.datetime_utc) and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM paypal
LIMIT 100;
"""
result_paypal = pd.read_sql(query_paypal, conn)
result_paypal

,type,status,currency,gross,fee,net,gross_usd,net_usd,provider_row_cnt
0,Payment Reversal,Reversed,EUR,-59.99,0.00,-59.99,-65.36,-65.36,1
1,Refund,Refunded,EUR,-354.83,0.00,-354.83,-385.62,-385.62,19
2,Refund,Refunded,GBP,-101.45,0.00,-101.45,-131.19,-131.19,7
3,Website Payment,Completed,EUR,28746.02,-1321.35,27424.67,31277.19,29839.79,1398
4,Website Payment,Completed,GBP,4259.41,-201.13,4058.28,5502.35,5242.72,259
5,Website Payment,Denied,EUR,2253.81,0.00,0.00,2453.18,0.00,119
6,Website Payment,Denied,GBP,506.73,0.00,0.00,655.31,0.00,27


In [246]:
query_paypal = """
WITH fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

paypal as (
    SELECT 
        pp.type,
        pp.status,
        pp.currency				
        , sum(CAST(REPLACE(pp.gross, ',', '.') AS REAL)) as gross
        , sum(CAST(REPLACE(pp.fee, ',', '.') AS REAL)) as fee
        , sum(CAST(REPLACE(pp.net, ',', '.') AS REAL)) as net
        , sum(ROUND(CAST(REPLACE(pp.gross, ',', '.') AS REAL) * fx.usd_rate, 2)) as gross_usd
        , sum(ROUND(CAST(REPLACE(pp.net, ',', '.') AS REAL) * fx.usd_rate, 2)) as net_usd
        , count(*) as provider_row_cnt
    FROM paypal_eu_activity_20260601_20260703 pp
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(pp.datetime_utc) 
        and date(pp.datetime_utc) < fx.next_rate_date
        and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM paypal
LIMIT 100;
"""
result_paypal = pd.read_sql(query_paypal, conn)
result_paypal

,type,status,currency,gross,fee,net,gross_usd,net_usd,provider_row_cnt
0,Payment Reversal,Reversed,EUR,-59.99,0.00,-59.99,-65.36,-65.36,1
1,Refund,Refunded,EUR,-354.83,0.00,-354.83,-385.62,-385.62,19
2,Refund,Refunded,GBP,-101.45,0.00,-101.45,-131.19,-131.19,7
3,Website Payment,Completed,EUR,28746.02,-1321.35,27424.67,31277.19,29839.79,1398
4,Website Payment,Completed,GBP,4259.41,-201.13,4058.28,5502.35,5242.72,259
5,Website Payment,Denied,EUR,2253.81,0.00,0.00,2453.18,0.00,119
6,Website Payment,Denied,GBP,506.73,0.00,0.00,655.31,0.00,27


In [207]:
query_paypal_eu_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
    , count(txn_id) as backend_row_cnt
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'paypal_eu'
GROUP BY 1,2,3
LIMIT 100;
"""
result_paypal_eu_backend = pd.read_sql(query_paypal_eu_backend, conn)
result_paypal_eu_backend

,status,operation_type,currency,amount_local,amount_usd,backend_row_cnt
0,declined,SALE,EUR,2253.81,2453.18,119
1,declined,SALE,GBP,506.73,655.31,27
2,settled,REFUND,EUR,329.84,358.63,18
3,settled,REFUND,GBP,101.45,131.19,7
4,settled,SALE,EUR,28775.99,31310.48,1401
5,settled,SALE,GBP,4259.41,5502.35,259


Discrepancies:
- Payment Reversal	Reversed	EUR: 1 row
- settled	REFUND	EUR: 329.84 vs 354.83	(+1 row)
- settled	SALE	EUR: 28746.02 vs 28775.99	(+3 rows)

In [202]:
query_paypal_eu_backend0 = """
SELECT *
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'paypal_eu'
    and currency = 'GBP'
    and operation_type = 'SALE'
    and status = 'declined' -- 'settled'
ORDER BY currency, created_at_utc
LIMIT 100;
"""
result_paypal_eu_backend0 = pd.read_sql(query_paypal_eu_backend0, conn)
result_paypal_eu_backend0

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-103463,ORD-503463,SALE,declined,paypal_eu,0DRQ0928NVSA52MA3,coach_addon,GB,GBP,4.49,1.284816,2026-06-01,5.77,2026-06-01T08:05:02Z,None
1,TXN-102959,ORD-502959,SALE,declined,paypal_eu,1VWNKJJ054C3NEPSX,sub_quarterly,GB,GBP,21.99,1.283249,2026-06-02,28.22,2026-06-02T10:54:04Z,None
2,TXN-102256,ORD-502256,SALE,declined,paypal_eu,YX0B37GEEPB9CWD3B,coach_addon,GB,GBP,4.49,1.282347,2026-06-03,5.76,2026-06-03T18:07:15Z,None
3,TXN-102829,ORD-502829,SALE,declined,paypal_eu,MGRAEKQTRV6282S9G,sub_monthly,GB,GBP,8.99,1.286215,2026-06-04,11.56,2026-06-04T16:31:32Z,None
4,TXN-103774,ORD-503774,SALE,declined,paypal_eu,22LA1JHF7VCWV3MT6,sub_monthly,GB,GBP,8.99,1.288623,2026-06-05,11.58,2026-06-05T08:09:00Z,None
5,TXN-102297,ORD-502297,SALE,declined,paypal_eu,A6S859QSEVLE56HR2,sub_monthly,GB,GBP,8.99,1.288623,2026-06-05,11.58,2026-06-07T13:13:40Z,None
6,TXN-103116,ORD-503116,SALE,declined,paypal_eu,5DUQ6WXZXH5KK7P6M,sub_monthly,GB,GBP,8.99,1.288623,2026-06-05,11.58,2026-06-07T15:07:13Z,None
7,TXN-103001,ORD-503001,SALE,declined,paypal_eu,E0M85AMZ9WCW8E0EP,sub_monthly,GB,GBP,8.99,1.290614,2026-06-08,11.60,2026-06-08T09:15:49Z,None
8,TXN-102871,ORD-502871,SALE,declined,paypal_eu,1NHSXXGLSVM0VJE8J,sub_monthly,GB,GBP,8.99,1.290138,2026-06-11,11.60,2026-06-11T08:01:17Z,None
9,TXN-103902,ORD-503902,SALE,declined,paypal_eu,C94GLQ9ESVB0CWKF5,sub_monthly,GB,GBP,8.99,1.293039,2026-06-12,11.62,2026-06-12T08:49:44Z,None


Discrepancies among settled + SALE + EUR:

In [253]:
query_paypal0 = """
WITH fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

paypal as (
    SELECT pp.*,
        CAST(REPLACE(pp.gross, ',', '.') AS REAL) as gross_fixed,
        ROUND(CAST(REPLACE(pp.gross, ',', '.') AS REAL) * fx.usd_rate, 2) as gross_usd
    FROM paypal_eu_activity_20260601_20260703 pp
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(pp.datetime_utc) 
        and date(pp.datetime_utc) < fx.next_rate_date
        and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
        and pp.type = 'Website Payment'
        and pp.status = 'Completed'
        and pp.currency = 'EUR'
    ORDER BY pp.datetime_utc
),

backend as (
    SELECT *
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'paypal_eu'
        and currency = 'EUR'
        and operation_type = 'SALE'
        and status = 'settled'
)

SELECT 
    paypal.datetime_utc,
    paypal.transaction_id,
    CAST(REPLACE(paypal.gross, ',', '.') AS REAL) as gross,
    backend.created_at_utc,
    backend.psp_reference,
    backend.amount_local
FROM paypal
FULL JOIN backend on backend.psp_reference = paypal.transaction_id
    -- backend.created_at_utc = paypal.datetime_utc
where CAST(REPLACE(paypal.gross, ',', '.') AS REAL) <> amount_local OR transaction_id is null or psp_reference is null
"""
result_paypal0 = pd.read_sql(query_paypal0, conn)
result_paypal0

,datetime_utc,transaction_id,gross,created_at_utc,psp_reference,amount_local
0,None,None,None,2026-06-03T12:00:00Z,N2D9QMQNH8VUEES5X,9.99
1,None,None,None,2026-06-09T12:00:00Z,TB26YQYEK9R2G8HQE,9.99
2,None,None,None,2026-06-12T12:00:00Z,GCV75356TRMG1HNWM,9.99


In [287]:
query_paypal2 = """
WITH paypal0 as (
    select distinct *
    from paypal_eu_activity_20260601_20260703
    where transaction_id in ('N2D9QMQNH8VUEES5X', 'TB26YQYEK9R2G8HQE', 'GCV75356TRMG1HNWM')
)
SELECT *
FROM paypal0
LIMIT 100;
"""
result_paypal2 = pd.read_sql(query_paypal2, conn)
result_paypal2

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,06/03/2026,12:20:00,GMT,Jürgen Müller,N2D9QMQNH8VUEES5X,None,Website Payment,Completed,EUR,"9,99","-0,64","9,35",ORD-507782,2026-03-06 12:20:00+00:00
1,06/09/2026,12:20:00,GMT,Émilie Laurent,TB26YQYEK9R2G8HQE,None,Website Payment,Completed,EUR,"9,99","-0,64","9,35",ORD-507783,2026-09-06 12:20:00+00:00
2,06/12/2026,12:20:00,GMT,Émilie Laurent,GCV75356TRMG1HNWM,None,Website Payment,Completed,EUR,"9,99","-0,64","9,35",ORD-507784,2026-12-06 12:20:00+00:00


Discrepancies among settled + REFUND + EUR:

In [256]:
query_paypal0 = """
WITH fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

paypal as (
    SELECT pp.*,
        CAST(REPLACE(pp.gross, ',', '.') AS REAL) as gross_fixed,
        ROUND(CAST(REPLACE(pp.gross, ',', '.') AS REAL) * fx.usd_rate, 2) as gross_usd
    FROM paypal_eu_activity_20260601_20260703 pp
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(pp.datetime_utc) 
        and date(pp.datetime_utc) < fx.next_rate_date
        and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
        and pp.type = 'Refund'
        and pp.status = 'Refunded'
        and pp.currency = 'EUR'
    ORDER BY pp.datetime_utc
),

backend as (
    SELECT *
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'paypal_eu'
        and currency = 'EUR'
        and operation_type = 'REFUND'
        and status = 'settled'
)

SELECT 
    paypal.datetime_utc,
    paypal.transaction_id,
    CAST(REPLACE(paypal.gross, ',', '.') AS REAL) as gross,
    backend.created_at_utc,
    backend.psp_reference,
    backend.amount_local
FROM paypal
FULL JOIN backend on backend.psp_reference = paypal.transaction_id
    -- backend.created_at_utc = paypal.datetime_utc
where -CAST(REPLACE(paypal.gross, ',', '.') AS REAL) <> amount_local OR transaction_id is null or psp_reference is null
"""
result_paypal0 = pd.read_sql(query_paypal0, conn)
result_paypal0

,datetime_utc,transaction_id,gross,created_at_utc,psp_reference,amount_local
0,2026-06-25 11:00:00+00:00,LCESPNBKZNLLA8UBX,-24.99,None,None,None


In [292]:
query_paypal2 = """
WITH paypal0 as (
    select distinct *
    from paypal_eu_activity_20260601_20260703
    where transaction_id in ('LCESPNBKZNLLA8UBX')
)
SELECT *
FROM paypal0
LIMIT 100;
"""
result_paypal2 = pd.read_sql(query_paypal2, conn)
result_paypal2

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,25/06/2026,11:00:00,GMT,Jürgen Müller,LCESPNBKZNLLA8UBX,YLU6F175STS5M7GG2,Refund,Refunded,EUR,"-24,99","0,00","-24,99",ORD-507806,2026-06-25 11:00:00+00:00


In [294]:
query_paypal2 = """
WITH paypal0 as (
    select distinct *
    from paypal_eu_activity_20260601_20260703
    where status = 'Reversed'
)
SELECT *
FROM paypal0
LIMIT 100;
"""
result_paypal2 = pd.read_sql(query_paypal2, conn)
result_paypal2

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,20/06/2026,13:00:00,GMT,José García,X2KQRFAWJMU8430TC,YHBCXCT8C0K378SHL,Payment Reversal,Reversed,EUR,"-59,99","0,00","-59,99",ORD-507788,2026-06-20 13:00:00+00:00


In [296]:
query_paypal2 = """
WITH paypal0 as (
    select distinct *
    from payment_engine_log
    where psp_reference in ('YLU6F175STS5M7GG2', 'YHBCXCT8C0K378SHL')
)
SELECT *
FROM paypal0
LIMIT 100;
"""
result_paypal2 = pd.read_sql(query_paypal2, conn)
result_paypal2

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107962,ORD-507788,SALE,settled,paypal_eu,YHBCXCT8C0K378SHL,sub_annual,ES,EUR,59.99,1.094662,2026-06-05,65.67,2026-06-05T09:30:00Z,2026-06-05T11:03:00Z
1,TXN-107980,ORD-507806,SALE,settled,paypal_eu,YLU6F175STS5M7GG2,sub_quarterly,DE,EUR,24.99,1.090715,2026-06-10,27.26,2026-06-10T10:00:00Z,2026-06-10T10:14:00Z


In [293]:
query_paypal2 = """
WITH paypal0 as (
    select distinct *
    from payment_engine_log
    where 1=1 
    --    and amount_local = 24.99 and currency = 'EUR'
    --    and date(created_at_utc) = '2026-06-25'
        and order_id = 'ORD-507806'
)
SELECT *
FROM paypal0
LIMIT 100;
"""
result_paypal2 = pd.read_sql(query_paypal2, conn)
result_paypal2

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107980,ORD-507806,SALE,settled,paypal_eu,YLU6F175STS5M7GG2,sub_quarterly,DE,EUR,24.99,1.090715,2026-06-10,27.26,2026-06-10T10:00:00Z,2026-06-10T10:14:00Z


Conclusion:
- 3 rows from backend and provider tables have different interpretation of date field: 06/03 <--> 03/06, 09/03 <--> 03/09, 12/03 <--> 03/12, so they're attributed to different months (9.99 x3 = 29.97 EUR difference)
- LCESPNBKZNLLA8UBX (YLU6F175STS5M7GG2) trxn has Refunded status in provider table but SALE+settled in backend table (24.99 EUR)
- X2KQRFAWJMU8430TC (YHBCXCT8C0K378SHL) trxn has Reversed status in provider table but SALE+settled in backend table (59.99 EUR)

In [357]:
query_paypal_eu_pivot_usd = """
WITH fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

paypal as (
    SELECT 
        pp.type,
        pp.status,
        pp.currency				
        , sum(CAST(REPLACE(pp.gross, ',', '.') AS REAL)) as gross
        , sum(CAST(REPLACE(pp.fee, ',', '.') AS REAL)) as fee
        , sum(CAST(REPLACE(pp.net, ',', '.') AS REAL)) as net
        , sum(ROUND(CAST(REPLACE(pp.gross, ',', '.') AS REAL) * fx.usd_rate, 2)) as gross_usd
        , sum(ROUND(CAST(REPLACE(pp.net, ',', '.') AS REAL) * fx.usd_rate, 2)) as net_usd
        , count(*) as provider_row_cnt
    FROM paypal_eu_activity_20260601_20260703 pp
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(pp.datetime_utc) 
        and date(pp.datetime_utc) < fx.next_rate_date
        and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT 
    'paypal_eu' as provider,
    sum(abs(gross_usd)) as gross_usd, 
    sum(provider_row_cnt) as provider_row_cnt
FROM paypal
"""
result_paypal_eu_pivot_usd = pd.read_sql(query_paypal_eu_pivot_usd, conn)
result_paypal_eu_pivot_usd

,provider,gross_usd,provider_row_cnt
0,paypal_eu,40470.2,1830


### PayPal US

In [283]:
query_paypal_us = """
WITH paypal0 as (
    select distinct *
    from paypal_us_activity_20260601_20260703
),

paypal as (
    SELECT 
        pp.type,
        pp.status,
        pp.currency				
        , sum(pp.gross) as gross
        , sum(pp.fee) as fee
        , sum(pp.net) as net
        -- , sum(pp.gross * fx.usd_rate) as gross_usd
        -- , sum(pp.net * fx.usd_rate) as net_usd
        , count(*) as provider_row_cnt
    FROM paypal0 pp
    -- LEFT JOIN fx_rates fx on fx.rate_date = date(pp.datetime_utc) and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM paypal
LIMIT 100;
"""
result_paypal_us = pd.read_sql(query_paypal_us, conn)
result_paypal_us

,type,status,currency,gross,fee,net,provider_row_cnt
0,Refund,Refunded,USD,-649.76,0.00,-649.76,39
1,Website Payment,Completed,USD,36721.06,-2208.46,34512.60,1894
2,Website Payment,Denied,USD,2923.42,0.00,0.00,158


In [284]:
query_paypal_us_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
    , count(txn_id) as backend_row_cnt
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'paypal_us'
GROUP BY 1,2,3
LIMIT 100;
"""
result_paypal_us_backend = pd.read_sql(query_paypal_us_backend, conn)
result_paypal_us_backend

,status,operation_type,currency,amount_local,amount_usd,backend_row_cnt
0,declined,SALE,USD,2923.42,2923.42,158
1,pending,SALE,USD,24.99,24.99,1
2,settled,REFUND,USD,649.76,649.76,39
3,settled,SALE,USD,36716.05,36716.05,1895


Discrepancies:
- settled	SALE: 36721.06 vs 	36731.05 (+1 row)
- pending	SALE: 24.99	vs None

In [285]:
query_paypal0 = """
WITH fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

paypal as (
    SELECT pp.*,
        CAST(REPLACE(pp.gross, ',', '.') AS REAL) as gross_fixed,
        ROUND(CAST(REPLACE(pp.gross, ',', '.') AS REAL) * fx.usd_rate, 2) as gross_usd
    FROM paypal_us_activity_20260601_20260703 pp
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(pp.datetime_utc) 
        and date(pp.datetime_utc) < fx.next_rate_date
        and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
        and pp.type = 'Website Payment'
        and pp.status = 'Completed'
    ORDER BY pp.datetime_utc
),

backend as (
    SELECT *
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'paypal_us'
        and operation_type = 'SALE'
        and status in ('settled') -- , 'pending'
)

SELECT 
    paypal.datetime_utc,
    paypal.transaction_id,
    CAST(REPLACE(paypal.gross, ',', '.') AS REAL) as gross,
    backend.created_at_utc,
    backend.psp_reference,
    backend.amount_local,
    backend.status
FROM paypal
FULL JOIN backend on backend.psp_reference = paypal.transaction_id
    -- backend.created_at_utc = paypal.datetime_utc
where ABS(CAST(REPLACE(paypal.gross, ',', '.') AS REAL)) <> amount_local OR transaction_id is null or psp_reference is null
"""
result_paypal0 = pd.read_sql(query_paypal0, conn)
result_paypal0

,datetime_utc,transaction_id,gross,created_at_utc,psp_reference,amount_local,status
0,2026-06-12 14:25:00+00:00,11EZAV8DU80K0J5GN,24.99,None,None,NaN,None
1,None,None,NaN,2026-06-30T23:59:20Z,LTUUWD3MW1R1X2KLT,9.99,settled
2,None,None,NaN,2026-06-30T23:59:57Z,GWSP0RPV10PGNPXA1,9.99,settled


In [286]:
query_paypal2 = """
WITH paypal0 as (
    select distinct *
    from paypal_us_activity_20260601_20260703
    where transaction_id in ('LTUUWD3MW1R1X2KLT', 'GWSP0RPV10PGNPXA1')
)
SELECT *
FROM paypal0
LIMIT 100;
"""
result_paypal2 = pd.read_sql(query_paypal2, conn)
result_paypal2

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,07/01/2026,00:04:10,GMT,Avery Collins,LTUUWD3MW1R1X2KLT,None,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-507780,2026-07-01 00:04:10+00:00
1,07/01/2026,00:05:10,GMT,Riley Carter,GWSP0RPV10PGNPXA1,None,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-507779,2026-07-01 00:05:10+00:00


In [261]:
query_paypal1 = """
    SELECT *
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'paypal_us'
        and operation_type = 'SALE'
        and status in ('pending')
"""
result_paypal1 = pd.read_sql(query_paypal1, conn)
result_paypal1

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107960,ORD-507786,SALE,pending,paypal_us,11EZAV8DU80K0J5GN,sub_quarterly,US,USD,24.99,1.0,2026-06-12,24.99,2026-06-12T14:00:00Z,None


Conclusion:
- backend table has 11EZAV8DU80K0J5GN trxn in pending status whereas provider table has it in settled status
- backend table has 2 more trxns: LTUUWD3MW1R1X2KLT and GWSP0RPV10PGNPXA1 (in provider tables they're logged in Jul, 01 already)

In [358]:
query_paypal_us_pivot_usd = """
WITH fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

paypal as (
    SELECT 
        pp.type,
        pp.status,
        pp.currency				
        , sum(CAST(REPLACE(pp.gross, ',', '.') AS REAL)) as gross
        , sum(CAST(REPLACE(pp.fee, ',', '.') AS REAL)) as fee
        , sum(CAST(REPLACE(pp.net, ',', '.') AS REAL)) as net
        --, sum(ROUND(CAST(REPLACE(pp.gross, ',', '.') AS REAL) * fx.usd_rate, 2)) as gross_usd
        --, sum(ROUND(CAST(REPLACE(pp.net, ',', '.') AS REAL) * fx.usd_rate, 2)) as net_usd
        , count(*) as provider_row_cnt
    FROM paypal_us_activity_20260601_20260703 pp
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(pp.datetime_utc) 
        and date(pp.datetime_utc) < fx.next_rate_date
        and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT 
    'paypal_us' as provider,
    sum(abs(gross)) as gross_usd, 
    sum(provider_row_cnt) as provider_row_cnt
FROM paypal
"""
result_paypal_us_pivot_usd = pd.read_sql(query_paypal_us_pivot_usd, conn)
result_paypal_us_pivot_usd

,provider,gross_usd,provider_row_cnt
0,paypal_us,40304.23,2092


### Adyen

In [350]:
query_adyen = """
WITH
fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),
adyen as (
    SELECT 
        aa.record_type,
        aa.gross_currency,
        aa.net_currency
        , sum(aa.gross_credit) as gross_credit
        , sum(aa.gross_debit) as gross_debit
        , sum(aa.commission) as commission
        , sum(aa.net_credit) as net_credit
        , sum(aa.net_debit) as net_debit
        , sum(round(aa.gross_credit * coalesce(fx.usd_rate, 1), 2)) as gross_credit_usd
        , sum(round(aa.gross_debit * coalesce(fx.usd_rate, 1), 2)) as gross_debit_usd
        , sum(round(aa.net_credit * coalesce(fx.usd_rate, 1), 2)) as net_credit_usd
        , sum(round(aa.net_debit * coalesce(fx.usd_rate, 1), 2)) as net_debit_usd
    FROM adyen_payment_accounting_20260601_20260703 aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.datetime_utc) 
        and date(aa.datetime_utc) < fx.next_rate_date
        and fx.currency = aa.gross_currency	
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM adyen
LIMIT 100;
"""
result_adyen = pd.read_sql(query_adyen, conn)
result_adyen

,record_type,gross_currency,net_currency,gross_credit,gross_debit,commission,net_credit,net_debit,gross_credit_usd,gross_debit_usd,net_credit_usd,net_debit_usd
0,Authorised,EUR,EUR,19389.92,NaN,NaN,NaN,NaN,21101.64,NaN,NaN,NaN
1,Authorised,GBP,GBP,4607.37,NaN,NaN,NaN,NaN,5952.25,NaN,NaN,NaN
2,Authorised,USD,USD,6276.94,NaN,NaN,NaN,NaN,6276.94,NaN,NaN,NaN
3,Chargeback,EUR,EUR,NaN,89.97,NaN,NaN,89.97,NaN,97.46,NaN,97.46
4,Refunded,EUR,EUR,NaN,272.37,NaN,NaN,272.37,NaN,295.90,NaN,295.90
5,Refunded,GBP,GBP,NaN,30.98,NaN,NaN,30.98,NaN,40.21,NaN,40.21
6,Refunded,USD,USD,NaN,49.97,NaN,NaN,49.97,NaN,49.97,NaN,49.97
7,Refused,EUR,EUR,2159.04,NaN,NaN,NaN,NaN,2349.99,NaN,NaN,NaN
8,Refused,GBP,GBP,302.86,NaN,NaN,NaN,NaN,391.20,NaN,NaN,NaN
9,Refused,USD,USD,614.76,NaN,NaN,NaN,NaN,614.76,NaN,NaN,NaN


In [88]:
query_adyen_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'adyen'
GROUP BY 1,2,3
LIMIT 100;
"""
result_adyen_backend = pd.read_sql(query_adyen_backend, conn)
result_adyen_backend

,status,operation_type,currency,amount_local,amount_usd
0,declined,SALE,EUR,2149.05,2339.09
1,declined,SALE,GBP,302.86,391.20
2,declined,SALE,USD,614.76,614.76
3,settled,CHARGEBACK,EUR,29.98,32.67
4,settled,REFUND,EUR,272.37,295.90
5,settled,REFUND,GBP,30.98,40.21
6,settled,REFUND,USD,49.97,49.97
7,settled,SALE,EUR,19389.92,21101.68
8,settled,SALE,GBP,4607.37,5952.22
9,settled,SALE,USD,6276.94,6276.94


In [302]:
query_adyen_reconciliation = """
WITH adyen_backend AS (
    SELECT 
        status, 
        operation_type, 
        currency
        , sum(amount_local) as amount_local
        , sum(amount_usd) as amount_usd
        , count(txn_id) as backend_row_cnt
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'adyen'
    GROUP BY 1,2,3
),

 fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),
adyen as (
    SELECT 
        CASE 
            WHEN aa.record_type = 'Authorised' THEN 'settled'
            WHEN aa.record_type = 'Chargeback' THEN 'settled'
            WHEN aa.record_type = 'Refunded' THEN 'settled'
            WHEN aa.record_type = 'Refused' THEN 'declined'
            -- WHEN aa.record_type = 'Settled' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.record_type = 'Authorised' THEN 'SALE'
            WHEN aa.record_type = 'Chargeback' THEN 'CHARGEBACK'
            WHEN aa.record_type = 'Refunded' THEN 'REFUND'
            WHEN aa.record_type = 'Refused' THEN 'SALE'
            -- WHEN aa.record_type = 'Settled' THEN 'SALE'
        END as operation_type,

        aa.gross_currency,
        aa.net_currency
        , sum(aa.gross_credit) as gross_credit
        , sum(aa.gross_debit) as gross_debit
        , sum(aa.commission) as commission
        , sum(aa.net_credit) as net_credit
        , sum(aa.net_debit) as net_debit
        , sum(ROUND(aa.gross_credit * coalesce(fx.usd_rate, 1), 2)) as gross_credit_usd
        , sum(ROUND(aa.gross_debit * coalesce(fx.usd_rate, 1), 2)) as gross_debit_usd
--        , sum(aa.net_credit * coalesce(fx.usd_rate, 1)) as net_credit_usd
--        , sum(aa.net_debit * coalesce(fx.usd_rate, 1)) as net_debit_usd
        , count(*) as provider_row_cnt
    FROM adyen_payment_accounting_20260601_20260703 aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.datetime_utc) 
        and date(aa.datetime_utc) < fx.next_rate_date
        and fx.currency = aa.gross_currency	
    -- LEFT JOIN fee_schedule fee on fee.account = aa.merchant_account
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3,4
)

SELECT 
    back.*,
    -- adyen.record_type,
    adyen.gross_currency,
    adyen.gross_credit,
    adyen.gross_credit_usd,
    adyen.gross_debit,
    adyen.gross_debit_usd,
    adyen.provider_row_cnt
FROM adyen_backend back
LEFT JOIN adyen ON adyen.operation_type = back.operation_type 
    AND adyen.status = back.status
    AND adyen.gross_currency = back.currency
"""
result_adyen_reconciliation = pd.read_sql(query_adyen_reconciliation, conn)
result_adyen_reconciliation

,status,operation_type,currency,amount_local,amount_usd,backend_row_cnt,gross_currency,gross_credit,gross_credit_usd,gross_debit,gross_debit_usd,provider_row_cnt
0,declined,SALE,EUR,2149.05,2339.09,95,EUR,2159.04,2349.99,NaN,NaN,96
1,declined,SALE,GBP,302.86,391.20,14,GBP,302.86,391.20,NaN,NaN,14
2,declined,SALE,USD,614.76,614.76,24,USD,614.76,614.76,NaN,NaN,24
3,settled,CHARGEBACK,EUR,29.98,32.67,2,EUR,NaN,NaN,89.97,97.46,3
4,settled,REFUND,EUR,272.37,295.90,18,EUR,NaN,NaN,272.37,295.90,18
5,settled,REFUND,GBP,30.98,40.21,2,GBP,NaN,NaN,30.98,40.21,2
6,settled,REFUND,USD,49.97,49.97,4,USD,NaN,NaN,49.97,49.97,4
7,settled,SALE,EUR,19389.92,21101.68,1008,EUR,19389.92,21101.64,NaN,NaN,1008
8,settled,SALE,GBP,4607.37,5952.22,263,GBP,4607.37,5952.25,NaN,NaN,263
9,settled,SALE,USD,6276.94,6276.94,306,USD,6276.94,6276.94,NaN,NaN,306


2 dicrepancies: 
- declined + SALE + EUR -- 2149.05 vs 2159.04 (+1 row)
- settled + CHARGEBACK + EUR -- 29.98 vs 89.97 (+1 row)

In [325]:
query_adyen_reconciliation0 = """
WITH adyen_backend AS (
    SELECT *
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'adyen'
    GROUP BY 1,2,3
),

fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),
adyen as (
    SELECT 
        CASE 
            WHEN aa.record_type = 'Authorised' THEN 'settled'
            WHEN aa.record_type = 'Chargeback' THEN 'settled'
            WHEN aa.record_type = 'Refunded' THEN 'settled'
            WHEN aa.record_type = 'Refused' THEN 'declined'
            -- WHEN aa.record_type = 'Settled' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.record_type = 'Authorised' THEN 'SALE'
            WHEN aa.record_type = 'Chargeback' THEN 'CHARGEBACK'
            WHEN aa.record_type = 'Refunded' THEN 'REFUND'
            WHEN aa.record_type = 'Refused' THEN 'SALE'
            -- WHEN aa.record_type = 'Settled' THEN 'SALE'
        END as operation_type,

        aa.*
        , coalesce(aa.gross_credit, aa.gross_debit) as amount
        , ROUND(aa.gross_credit * coalesce(fx.usd_rate, 1), 2) as gross_credit_usd
        , ROUND(aa.gross_debit * coalesce(fx.usd_rate, 1), 2) as gross_debit_usd
    FROM adyen_payment_accounting_20260601_20260703 aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.datetime_utc) 
        and date(aa.datetime_utc) < fx.next_rate_date
        and fx.currency = aa.gross_currency	
    -- LEFT JOIN fee_schedule fee on fee.account = aa.merchant_account
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
)

SELECT 
    provider.datetime_utc,
    provider.record_type,
    provider.gross_currency,
    provider.amount,
    backend.created_at_utc,
    backend.status,
    backend.operation_type,
    backend.currency,
    backend.amount_local
FROM adyen provider
FULL JOIN adyen_backend backend on backend.amount_local = provider.amount
    and datetime(backend.created_at_utc) = datetime(provider.datetime_utc)
--where provider.record_type = 'Refused' and provider.gross_currency = 'EUR'
where provider.record_type = 'Chargeback' and provider.gross_currency = 'EUR'

-- backend.status = 'declined' and backend.operation_type = 'SALE' and backend.currency = 'EUR'
"""
result_adyen_reconciliation0 = pd.read_sql(query_adyen_reconciliation0, conn)
result_adyen_reconciliation0[0:30]

,datetime_utc,record_type,gross_currency,amount,created_at_utc,status,operation_type,currency,amount_local
0,2026-06-19 10:30:59+00:00,Chargeback,EUR,4.99,2026-06-19T10:30:59Z,settled,CHARGEBACK,EUR,4.99
1,2026-06-21 10:30:30+00:00,Chargeback,EUR,24.99,2026-06-21T10:30:30Z,settled,CHARGEBACK,EUR,24.99
2,2026-06-26 14:00:00+00:00,Chargeback,EUR,59.99,None,None,None,None,NaN


Conclusion:
- trxn at 2026-06-08 11:00:00+00:00 (9.99 EUR) -- Refused in provider table, settled (not declined) in backend table
- trxn at 2026-06-26 14:00:00+00:00	(59.99 EUR) -- Chargeback in provider table, not in backend table

In [359]:
query_adyen_pivot_usd = """
WITH
fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),
adyen as (
    SELECT 
        aa.record_type,
        aa.gross_currency,
        aa.net_currency
        , sum(aa.gross_credit) as gross_credit
        , sum(aa.gross_debit) as gross_debit
        , sum(aa.commission) as commission
        , sum(aa.net_credit) as net_credit
        , sum(aa.net_debit) as net_debit
        , sum(round(aa.gross_credit * coalesce(fx.usd_rate, 1), 2)) as gross_credit_usd
        , sum(round(aa.gross_debit * coalesce(fx.usd_rate, 1), 2)) as gross_debit_usd
        , sum(round(aa.net_credit * coalesce(fx.usd_rate, 1), 2)) as net_credit_usd
        , sum(round(aa.net_debit * coalesce(fx.usd_rate, 1), 2)) as net_debit_usd
        , count(*) as provider_row_cnt
    FROM adyen_payment_accounting_20260601_20260703 aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.datetime_utc) 
        and date(aa.datetime_utc) < fx.next_rate_date
        and fx.currency = aa.gross_currency	
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT 
    'adyen' as provider,
    sum(gross_credit_usd) + sum(gross_debit_usd) as gross_usd,
    sum(provider_row_cnt) as provider_row_cnt
FROM adyen
where record_type != 'Settled'
"""
result_adyen_pivot_usd = pd.read_sql(query_adyen_pivot_usd, conn)
result_adyen_pivot_usd

,provider,gross_usd,provider_row_cnt
0,adyen,37170.32,1738


### dLocal

In [360]:
query_dlocal = """
WITH dlocal0 as (
    select distinct * 
    from dlocal_transactions_20260601_20260703
),

dlocal as (
    SELECT 
        transaction_type,
        status,
        country,
        currency
        , sum(round(local_amount, 2) / 100) as local_amount
        , sum(usd_amount) as usd_amount
        , count(*) as provider_row_cnt
    FROM dlocal0
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    GROUP BY 1,2,3,4
)

SELECT *
FROM dlocal
LIMIT 100;
"""
result_dlocal = pd.read_sql(query_dlocal, conn)
result_dlocal

,transaction_type,status,country,currency,local_amount,usd_amount,provider_row_cnt
0,PAYMENT,IN_MEDIATION,BR,BRL,34.90,6.45,1
1,PAYMENT,PAID,AR,ARS,1170500.00,978.41,100
2,PAYMENT,PAID,BR,BRL,35360.00,6517.64,510
3,PAYMENT,PAID,CL,CLP,18756.60,1992.40,134
4,PAYMENT,PAID,CO,COP,7291800.00,1797.63,122
5,PAYMENT,PAID,MX,MXN,70266.00,3876.26,284
6,PAYMENT,REJECTED,AR,ARS,115992.00,96.92,8
7,PAYMENT,REJECTED,BR,BRL,3901.50,720.13,55
8,PAYMENT,REJECTED,CL,CLP,1668.70,177.13,13
9,PAYMENT,REJECTED,CO,COP,599300.00,148.95,7


In [298]:
query_dlocal_backend = """
SELECT 
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'dlocal'
GROUP BY 1,2,3
LIMIT 100;
"""
result_dlocal_backend = pd.read_sql(query_dlocal_backend, conn)
result_dlocal_backend

,status,operation_type,currency,amount_local,amount_usd
0,declined,SALE,ARS,115992.00,96.92
1,declined,SALE,BRL,3901.50,720.13
2,declined,SALE,CLP,166870.00,177.13
3,declined,SALE,COP,599300.00,148.95
4,declined,SALE,MXN,6834.00,377.29
5,settled,REFUND,ARS,13998.50,11.67
6,settled,REFUND,BRL,949.97,174.91
7,settled,REFUND,CLP,7485.00,7.94
8,settled,REFUND,MXN,329.00,18.11
9,settled,SALE,ARS,1170500.00,978.39


In [299]:
query_dlocal_reconciliation = """
WITH dlocal_backend AS (
    SELECT 
        -- sku??
        status, 
        operation_type, 
        currency
        , sum(amount_local) as amount_local
        , sum(amount_usd) as amount_usd
        , count(txn_id) as backend_row_cnt
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'dlocal'
    GROUP BY 1,2,3
),

dlocal0 as (
    select distinct * 
    from dlocal_transactions_20260601_20260703
),
dlocal as (
    SELECT 
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'settled'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'declined'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'REFUND'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'SALE'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'SALE'
        END as operation_type,

        aa.transaction_type,
        aa.status as dlocal_status,
        currency
        , sum(round(local_amount, 2) / 100) as local_amount
        , sum(usd_amount) as usd_amount
        , sum(usd_amount * fx_rate) as fx_
        , count(transaction_id) as provider_row_cnt
    FROM dlocal0 aa
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    GROUP BY 1,2,3,4,5
)

SELECT 
    back.*,
    dlocal.transaction_type,
    dlocal.dlocal_status,
    dlocal.currency,
    dlocal.local_amount,
    dlocal.usd_amount,
    dlocal.provider_row_cnt
FROM dlocal_backend back
FULL JOIN dlocal ON dlocal.operation_type = back.operation_type 
    AND dlocal.status = back.status
    AND dlocal.currency = back.currency
"""
result_dlocal_reconciliation = pd.read_sql(query_dlocal_reconciliation, conn)
result_dlocal_reconciliation

,status,operation_type,currency,amount_local,amount_usd,backend_row_cnt,transaction_type,dlocal_status,currency,local_amount,usd_amount,provider_row_cnt
0,declined,SALE,ARS,115992.00,96.92,8.0,PAYMENT,REJECTED,ARS,115992.00,96.92,8
1,declined,SALE,BRL,3901.50,720.13,55.0,PAYMENT,REJECTED,BRL,3901.50,720.13,55
2,declined,SALE,CLP,166870.00,177.13,13.0,PAYMENT,REJECTED,CLP,1668.70,177.13,13
3,declined,SALE,COP,599300.00,148.95,7.0,PAYMENT,REJECTED,COP,599300.00,148.95,7
4,declined,SALE,MXN,6834.00,377.29,26.0,PAYMENT,REJECTED,MXN,6834.00,377.29,26
5,settled,REFUND,ARS,13998.50,11.67,2.0,REFUND,REFUNDED,ARS,13998.50,11.67,2
6,settled,REFUND,BRL,949.97,174.91,13.0,REFUND,REFUNDED,BRL,989.95,182.27,13
7,settled,REFUND,CLP,7485.00,7.94,2.0,REFUND,REFUNDED,CLP,74.85,7.94,2
8,settled,REFUND,MXN,329.00,18.11,1.0,REFUND,REFUNDED,MXN,329.00,18.11,1
9,settled,SALE,ARS,1170500.00,978.39,100.0,PAYMENT,PAID,ARS,1170500.00,978.41,100


Dicrepancies:

- "not divided by 100" case:
    - declined + SALE + CLP -- 166870.00 vs 1668.70 (usd ok)
    - settled	REFUND	CLP	-- 7485.00 vs 74.85 (usd ok)

- others:
    - settled	REFUND	BRL	-- 949.97 vs 989.95 (usd TOO)
    - settled	SALE	ARS -- USD 978.39 vs 978.41
    - settled	SALE	BRL	-- 35394.90	vs 35360.00	(usd TOO) - (-1 row)
    - settled	SALE	CLP	-- 1882650.00 vs 18756.60	(usd TOO) - (-1 row)
    - settled	SALE	COP	-- USD 201648.85 vs 1797.63 (local ok)
    - settled	SALE	MXN	-- 70137.00 vs 70266.00	(usd TOO) - (+1 row)
    - PAYMENT	IN_MEDIATION	BRL -- there is no backend

In [379]:
query_dlocal0 = """
WITH dlocal0 as (
    select distinct * 
    from dlocal_transactions_20260601_20260703
),
fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

dlocal as (
    SELECT 
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'settled'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'declined'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'REFUND'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'SALE'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'SALE'
        END as operation_type,

        aa.*
        , round(aa.local_amount * fx.usd_rate, 2) as usd_amount_recalc
        , aa.status as dlocal_status
    FROM dlocal0 aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.created_at_utc) 
        and date(aa.created_at_utc) < fx.next_rate_date
        and fx.currency = aa.currency	
    WHERE aa.created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
),

backend as (
    SELECT aa.*
        , round(aa.amount_local * fx.usd_rate, 2) as amount_usd_recalc
    FROM payment_engine_log aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.created_at_utc) 
        and date(aa.created_at_utc) < fx.next_rate_date
        and fx.currency = aa.currency	
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'dlocal'
        --and currency = 'EUR'
        --and operation_type = 'SALE'
        --and status = 'settled'
)

SELECT 
    provider.transaction_id,
--    provider.status,
    provider.transaction_type,
    provider.dlocal_status,
--    provider.operation_type,
    provider.created_at_utc,
    provider.currency,
    provider.local_amount,
    provider.usd_amount,
    provider.usd_amount_recalc,

    backend.operation_type,
    backend.status,
    backend.psp_reference,
    backend.created_at_utc,
    backend.psp_reference,
    backend.amount_local,
    backend.amount_usd,
    backend.amount_usd_recalc
FROM dlocal provider
FULL JOIN backend on backend.psp_reference = provider.transaction_id
    -- backend.created_at_utc = paypal.datetime_utc
where NOT (CAST(REPLACE(provider.local_amount, ',', '.') AS REAL) = amount_local 
            OR CAST(REPLACE(provider.local_amount, ',', '.') AS REAL) / 100 = amount_local
            )
    OR provider.usd_amount <> backend.amount_usd
    OR transaction_id is null or psp_reference is null
"""
result_dlocal0 = pd.read_sql(query_dlocal0, conn)
result_dlocal0

,transaction_id,transaction_type,dlocal_status,created_at_utc,currency,local_amount,usd_amount,usd_amount_recalc,operation_type,status,psp_reference,created_at_utc,psp_reference,amount_local,amount_usd,amount_usd_recalc
0,DL-90001309,PAYMENT,PAID,2026-06-01 00:05:00,MXN,12900.0,7.11,709.68,None,None,None,None,None,NaN,NaN,NaN
1,DL-90001318,PAYMENT,PAID,2026-06-05 10:03:00,ARS,1599900.0,13.45,1343.92,SALE,settled,DL-90001318,2026-06-05T09:30:00Z,DL-90001318,15999.00,13.44,13.44
2,DL-90001311,PAYMENT,PAID,2026-06-09 10:08:00,BRL,8990.0,16.60,1659.99,SALE,settled,DL-90001311,2026-06-09T10:00:00Z,DL-90001311,89.90,89.90,16.60
3,DL-90001308,PAYMENT,PAID,2026-06-10 15:33:00,BRL,8990.0,17.13,1712.97,SALE,settled,DL-90001308,2026-06-10T15:00:00Z,DL-90001308,89.90,16.66,17.13
4,DL-90001312,PAYMENT,PAID,2026-06-15 11:49:00,MXN,79900.0,44.08,4408.00,SALE,settled,DL-90001312,2026-06-15T10:00:00Z,DL-90001312,799.00,799.00,44.08
5,DL-90001315,REFUND,REFUNDED,2026-06-18 12:00:00,BRL,9995.0,18.40,1839.55,REFUND,settled,DL-90001315,2026-06-18T12:00:00Z,DL-90001315,59.97,11.04,11.04
6,DL-90001313,PAYMENT,PAID,2026-06-22 11:36:00,COP,19990000.0,48.78,4877.56,SALE,settled,DL-90001313,2026-06-22T10:00:00Z,DL-90001313,199900.00,199900.00,48.78
7,DL-90001319,PAYMENT,PAID,2026-06-23 09:45:00,ARS,1599900.0,13.37,1335.92,SALE,settled,DL-90001319,2026-06-23T09:30:00Z,DL-90001319,15999.00,13.36,13.36
8,None,None,None,None,None,NaN,NaN,NaN,SALE,settled,DL-90001317,2026-06-20T12:00:00Z,DL-90001317,6990.00,7.42,7.42


Conclusion:
- trxn DL-90001311, DL-90001312, DL-90001313: in backend table, amount_usd was accidentally filled with the same value as amount_local (89.90, 799.00, 199900.00 instead of 16.60, 44.08, 48.78)
- trxn DL-90001317 is not in provider table
- trxn DL-90001309 is not in backend table
- trxn DL-90001315 has different amounts (in BRL, in USD too): 9995.0 in provider, 59.97 in backend
- trxn DL-90001318, DL-90001319 have different USD rounding (13.45 vs 13.44, 13.37	vs 13.36)
- trxn DL-90001308 has different conversion to USD (17.13 vs 16.66 -- the last is wrong)
- 1127 TRXNs: local_amount and amount_local differs in 100 times (local_amount was not divided by 100 in many cases)

In [382]:
query_dlocal100 = """
WITH dlocal0 as (
    select distinct * 
    from dlocal_transactions_20260601_20260703
),
fx_rates_fixed as (
    select *
        , row_number() over (partition by rate_date, currency order by published_at desc) as rn
    from fx_rates
),
fx_rates_2 as (
    select *, lead(rate_date) over (partition by currency order by rate_date) as next_rate_date
    from fx_rates_fixed
    where rn=1
    order by currency, rate_date
),

dlocal as (
    SELECT 
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'settled'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'declined'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'REFUND'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'SALE'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'SALE'
        END as operation_type,

        aa.*
        , round(aa.local_amount * fx.usd_rate, 2) as usd_amount_recalc
        , aa.status as dlocal_status
    FROM dlocal0 aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.created_at_utc) 
        and date(aa.created_at_utc) < fx.next_rate_date
        and fx.currency = aa.currency	
    WHERE aa.created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
),

backend as (
    SELECT aa.*
        , round(aa.amount_local * fx.usd_rate, 2) as amount_usd_recalc
    FROM payment_engine_log aa
    LEFT JOIN fx_rates_2 fx on fx.rate_date <= date(aa.created_at_utc) 
        and date(aa.created_at_utc) < fx.next_rate_date
        and fx.currency = aa.currency	
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'dlocal'
        --and currency = 'EUR'
        --and operation_type = 'SALE'
        --and status = 'settled'
)

SELECT 
    provider.transaction_id,
--    provider.status,
    provider.transaction_type,
    provider.dlocal_status,
--    provider.operation_type,
    provider.created_at_utc,
    provider.currency,
    provider.local_amount,
    provider.usd_amount,
    provider.usd_amount_recalc,

    backend.operation_type,
    backend.status,
    backend.psp_reference,
    backend.created_at_utc,
    backend.psp_reference,
    backend.amount_local,
    backend.amount_usd,
    backend.amount_usd_recalc
FROM dlocal provider
FULL JOIN backend on backend.psp_reference = provider.transaction_id
where CAST(REPLACE(provider.local_amount, ',', '.') AS REAL) / 100 = backend.amount_local
"""
result_dlocal100 = pd.read_sql(query_dlocal100, conn)
result_dlocal100

,transaction_id,transaction_type,dlocal_status,created_at_utc,currency,local_amount,usd_amount,usd_amount_recalc,operation_type,status,psp_reference,created_at_utc,psp_reference,amount_local,amount_usd,amount_usd_recalc
0,DL-90001242,PAYMENT,REJECTED,2026-06-01 07:25:52,COP,19990000,50.17,5017.49,SALE,declined,DL-90001242,2026-06-01T07:25:52Z,DL-90001242,199900.0,50.17,50.17
1,DL-90000404,PAYMENT,PAID,2026-06-01 08:17:24,MXN,12900,7.10,709.68,SALE,settled,DL-90000404,2026-06-01T07:11:24Z,DL-90000404,129.0,7.10,7.10
2,DL-90000750,PAYMENT,PAID,2026-06-01 08:27:11,COP,2990000,7.50,750.49,SALE,settled,DL-90000750,2026-06-01T07:11:11Z,DL-90000750,29900.0,7.50,7.50
3,DL-90001024,PAYMENT,PAID,2026-06-01 08:29:13,BRL,3490,6.44,643.97,SALE,settled,DL-90001024,2026-06-01T08:17:13Z,DL-90001024,34.9,6.44,6.44
4,DL-90000922,PAYMENT,PAID,2026-06-01 08:43:54,BRL,3490,6.44,643.97,SALE,settled,DL-90000922,2026-06-01T08:18:54Z,DL-90000922,34.9,6.44,6.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1122,DL-90000567,PAYMENT,REJECTED,2026-06-30 18:54:18,BRL,3490,6.38,638.13,SALE,declined,DL-90000567,2026-06-30T18:54:18Z,DL-90000567,34.9,6.38,6.38
1123,DL-90000170,PAYMENT,PAID,2026-06-30 18:54:35,BRL,19990,36.55,3655.05,SALE,settled,DL-90000170,2026-06-30T17:55:35Z,DL-90000170,199.9,36.55,36.55
1124,DL-90000386,PAYMENT,PAID,2026-06-30 19:23:59,MXN,12900,7.10,710.26,SALE,settled,DL-90000386,2026-06-30T18:05:59Z,DL-90000386,129.0,7.10,7.10
1125,DL-90000164,PAYMENT,PAID,2026-06-30 19:45:15,COP,2990000,7.36,735.54,SALE,settled,DL-90000164,2026-06-30T18:17:15Z,DL-90000164,29900.0,7.36,7.36


In [361]:
query_dlocal_pivot_usd = """
WITH dlocal0 as (
    select distinct * 
    from dlocal_transactions_20260601_20260703
),

dlocal as (
    SELECT 
        transaction_type,
        status,
        country,
        currency
        , sum(round(local_amount, 2) / 100) as local_amount
        , sum(usd_amount) as usd_amount
        , count(*) as provider_row_cnt
    FROM dlocal0
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    GROUP BY 1,2,3,4
)

SELECT 
    'dlocal' as provider,
    sum(usd_amount) as gross_usd,
    sum(provider_row_cnt) as provider_row_cnt
FROM dlocal
"""
result_dlocal_pivot_usd = pd.read_sql(query_dlocal_pivot_usd, conn)
result_dlocal_pivot_usd

,provider,gross_usd,provider_row_cnt
0,dlocal,16909.2,1278


### Google Play

In [104]:
query_googleplay = """
WITH googleplay_0 as (
    SELECT * FROM google_play_earnings_202606
    UNION ALL
    SELECT * FROM google_play_earnings_202607_partial
),

googleplay as (
    SELECT 
        transaction_type,
        buyer_currency,
        merchant_currency
        , sum(amount_buyer_currency) as amount_buyer_currency
        , sum(amount_merchant_currency) as amount_merchant_currency
    FROM googleplay_0
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM googleplay
LIMIT 100;
"""
result_googleplay = pd.read_sql(query_googleplay, conn)
result_googleplay

,transaction_type,buyer_currency,merchant_currency,amount_buyer_currency,amount_merchant_currency
0,Charge,BRL,USD,8806.70,1622.49
1,Charge,EUR,USD,3438.49,3745.96
2,Charge,GBP,USD,1963.95,2538.65
3,Charge,MXN,USD,17698.00,976.39
4,Charge,USD,USD,6046.85,6046.85
5,Charge refund,BRL,USD,-34.90,-6.44
6,Charge refund,EUR,USD,-79.96,-86.87
7,Charge refund,GBP,USD,-21.99,-28.54
8,Charge refund,MXN,USD,-799.00,-43.89
9,Charge refund,USD,USD,-19.97,-19.97


In [105]:
query_googleplay_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'google_play'
GROUP BY 1,2,3
LIMIT 100;
"""
result_googleplay_backend = pd.read_sql(query_googleplay_backend, conn)
result_googleplay_backend

,status,operation_type,currency,amount_local,amount_usd
0,settled,REFUND,BRL,34.90,6.44
1,settled,REFUND,EUR,79.96,86.87
2,settled,REFUND,GBP,21.99,28.54
3,settled,REFUND,MXN,799.00,43.89
4,settled,REFUND,USD,19.97,19.97
5,settled,SALE,BRL,8806.70,1622.49
6,settled,SALE,EUR,3438.49,3745.96
7,settled,SALE,GBP,1963.95,2538.65
8,settled,SALE,MXN,17698.00,976.39
9,settled,SALE,USD,6046.85,6046.85


In [122]:
query_googleplay_reconciliation = """
WITH googleplay_backend AS (
    SELECT 
        -- sku??
        status, 
        operation_type, 
        currency
        , sum(amount_local) as amount_local
        , sum(amount_usd) as amount_usd
        , count(txn_id) as backend_row_cnt
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'google_play'
    GROUP BY 1,2,3
),

googleplay_0 as (
    SELECT * FROM google_play_earnings_202606
    UNION ALL
    SELECT * FROM google_play_earnings_202607_partial
),

googleplay as (
    SELECT 
        CASE 
            WHEN aa.transaction_type = 'Charge' THEN 'settled'
            WHEN aa.transaction_type = 'Charge refund' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.transaction_type = 'Charge' THEN 'SALE'
            WHEN aa.transaction_type = 'Charge refund' THEN 'REFUND'
        END as operation_type,

        transaction_type,
        buyer_currency,
        merchant_currency
        , sum(amount_buyer_currency) as amount_buyer_currency
        , sum(amount_merchant_currency) as amount_merchant_currency
        , count(*) as provider_row_cnt
    FROM googleplay_0 aa
--     LEFT JOIN fee_schedule fee on 1=1 and fee.psp = 'google_play'
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3,4
)

SELECT 
    back.*,
    googleplay.transaction_type,
    googleplay.amount_buyer_currency,
    googleplay.amount_merchant_currency,
    googleplay.provider_row_cnt
FROM googleplay_backend back
FULL JOIN googleplay ON googleplay.operation_type = back.operation_type 
    AND googleplay.status = back.status
    AND googleplay.buyer_currency = back.currency
"""
result_googleplay_reconciliation = pd.read_sql(query_googleplay_reconciliation, conn)
result_googleplay_reconciliation

,status,operation_type,currency,amount_local,amount_usd,backend_row_cnt,transaction_type,amount_buyer_currency,amount_merchant_currency,provider_row_cnt
0,settled,REFUND,BRL,34.90,6.44,1.0,Charge refund,-34.90,-6.44,1
1,settled,REFUND,EUR,79.96,86.87,4.0,Charge refund,-79.96,-86.87,4
2,settled,REFUND,GBP,21.99,28.54,1.0,Charge refund,-21.99,-28.54,1
3,settled,REFUND,MXN,799.00,43.89,1.0,Charge refund,-799.00,-43.89,1
4,settled,REFUND,USD,19.97,19.97,3.0,Charge refund,-19.97,-19.97,3
5,settled,SALE,BRL,8806.70,1622.49,123.0,Charge,8806.70,1622.49,123
6,settled,SALE,EUR,3438.49,3745.96,151.0,Charge,3438.49,3745.96,151
7,settled,SALE,GBP,1963.95,2538.65,105.0,Charge,1963.95,2538.65,105
8,settled,SALE,MXN,17698.00,976.39,72.0,Charge,17698.00,976.39,72
9,settled,SALE,USD,6046.85,6046.85,315.0,Charge,6046.85,6046.85,315


Conclusion:
- No discrepancies in both currencies

In [366]:
query_googleplay_pivot_usd = """
WITH googleplay_0 as (
    SELECT * FROM google_play_earnings_202606
    UNION ALL
    SELECT * FROM google_play_earnings_202607_partial
),

googleplay as (
    SELECT 
        transaction_type,
        buyer_currency,
        merchant_currency
        , sum(amount_buyer_currency) as amount_buyer_currency
        , sum(amount_merchant_currency) as amount_merchant_currency
        , count(*) as provider_row_cnt
    FROM googleplay_0
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT 
    'google_play' as provider,
    sum(abs(amount_merchant_currency)) as gross_usd,
    sum(provider_row_cnt) as provider_row_cnt
FROM googleplay
where transaction_type != 'Google fee'
"""
result_googleplay_pivot_usd = pd.read_sql(query_googleplay_pivot_usd, conn)
result_googleplay_pivot_usd

,provider,gross_usd,provider_row_cnt
0,google_play,15116.05,776


## NOTES

1. Each of **paypal_us_activity_20260601_20260703** and **dlocal_transactions_20260601_20260703** has 1 duplicated row
2. FX rates table has 2 versions of BRL for Jun 10, so I used the latest value
3. FX rates table doesn't have data for weekends, so I used the rates for the last workday (usualy Friday)
4. In Google Play data, I don't use transaction_type = 'Google fee' for comparison because there is no such type on the backend side
5. In Adyen data, I don't use 'Settled' transaction type because this data duplicates that one from 'Authorised' type
6. I added a new column for almost all provider tables -- **datetime_utc** (creation date in UTC timezone) to compare data with backend in the same timezone. Some tables (dlocal and backend) already had the analogue: **created_at_utc**

## DISCREPANCIES (in USD and rows)

In [342]:
result_backend_pivot_usd

,psp,amount_usd,backend_row_cnt
0,adyen,37094.64,1736
1,dlocal,217581.10,1278
2,google_play,15116.05,776
3,paypal_eu,40411.14,1831
4,paypal_us,40314.22,2093


In [368]:
result_providers_pivot_usd = pd.concat([
    result_adyen_pivot_usd, 
    result_dlocal_pivot_usd,
    result_googleplay_pivot_usd,
    result_paypal_eu_pivot_usd, 
    result_paypal_us_pivot_usd
]).reset_index(drop=True)

result_providers_pivot_usd

,provider,gross_usd,provider_row_cnt
0,adyen,37170.32,1738
1,dlocal,16909.20,1278
2,google_play,15116.05,776
3,paypal_eu,40470.20,1830
4,paypal_us,40304.23,2092


In [373]:
df_diff = result_providers_pivot_usd.merge(result_backend_pivot_usd, left_on='provider', right_on='psp').drop(columns=['psp'])
df_diff['usd_diff'] = df_diff['gross_usd'].round(2) - df_diff['amount_usd'].round(2)
df_diff['row_cnt_diff'] = df_diff['provider_row_cnt'] - df_diff['backend_row_cnt']
df_diff

,provider,gross_usd,provider_row_cnt,amount_usd,backend_row_cnt,usd_diff,row_cnt_diff
0,adyen,37170.32,1738,37094.64,1736,75.68,2
1,dlocal,16909.20,1278,217581.10,1278,-200671.90,0
2,google_play,15116.05,776,15116.05,776,0.00,0
3,paypal_eu,40470.20,1830,40411.14,1831,59.06,-1
4,paypal_us,40304.23,2092,40314.22,2093,-9.99,-1


In [376]:
round(float(df_diff["usd_diff"].abs().sum()), 2)

200816.63

In [377]:
int(df_diff["row_cnt_diff"].abs().sum())

4

In [378]:
199900.00 / 200816.3

0.9954371233809208

We have 4 discrepancies in rows (2 in Adyen, 2 in PayPal) and 200816.3 USD, but 99,5% of the last discrepancy came from the dLocal cause where USD value was erroneously replaced with value from local currency (199900.00 instead of 48.78 USD)